In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("data/EDA_ML/financial_data.csv")

In [3]:
cluster_df = df.groupby('symbol').agg({
    'net_profit_margin_pct': 'mean',
    'sales': 'mean',
    'debt_to_equity': 'mean',
    'cash_conversion_ratio': 'mean',
    'dividend_payout_pct': 'mean',
    'return_on_assets': 'mean',
    'company_name': 'first'
}).reset_index()

In [4]:
from sklearn.preprocessing import StandardScaler

features = [
    'net_profit_margin_pct',
    'sales',
    'debt_to_equity',
    'cash_conversion_ratio',
    'dividend_payout_pct',
    'return_on_assets'
]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_df[features])

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(X_scaled)

In [6]:
sim_df = pd.DataFrame(
    similarity_matrix,
    index=cluster_df['symbol'],
    columns=cluster_df['symbol']
)

In [7]:
def get_top_peers(company_symbol, top_n=5):
    scores = sim_df.loc[company_symbol]
    
    # remove self
    scores = scores.drop(company_symbol)
    
    top_peers = scores.sort_values(ascending=False).head(top_n)
    
    return top_peers

In [8]:
get_top_peers('TCS')

symbol
NTPC         0.948588
INFY         0.944941
M&M          0.903190
COALINDIA    0.899274
GAIL         0.848781
Name: TCS, dtype: float64

In [9]:
peer_rows = []

for company in sim_df.index:
    top_peers = get_top_peers(company)
    
    for peer, score in top_peers.items():
        peer_rows.append({
            'company': company,
            'peer': peer,
            'similarity_score': score
        })

peer_df = pd.DataFrame(peer_rows)

In [10]:
peer_df.head(10)

,company,peer,similarity_score
0,ABB,TORNTPHARM,0.944561
1,ABB,DLF,0.934284
2,ABB,POWERGRID,0.925140
3,ABB,AMBUJACEM,0.923273
4,ABB,BRITANNIA,0.916288
5,ADANIENSOL,TATACONSUM,0.929121
6,ADANIENSOL,DMART,0.927857
7,ADANIENSOL,KOTAKBANK,0.924100
8,ADANIENSOL,ADANIENT,0.919944
9,ADANIENSOL,LODHA,0.912213
